# Domača naloga 1: Postopno reševanje (z razlago)

Ta zvezek vodi skozi celotno nalogo korak za korakom:
- kako modeliramo obremenitev,
- zakaj izberemo določene enačbe,
- kako dobimo notranje sile in navore,
- kako narišemo skice in NTM diagrame,
- kako preverimo pravilnost rezultata.

## 1) Podatki naloge in koordinatni sistem

Uporabimo desnosučni koordinatni sistem:
- os x: vzdolž stebla konzole od vpetja proti prostemu koncu,
- os y: radialna smer,
- os z: obodna smer.

Podani podatki:
- $L = 34.75\ \mathrm{mm}$
- $r = 17.5\ \mathrm{mm}$
- $F_a = -35.5\ \mathrm{N}$ (aksialna)
- $F_r = 37.25\ \mathrm{N}$ (radialna)
- $F_c = 113\ \mathrm{N}$ (obodna)

Negativen predznak pri $F_a$ pomeni, da deluje v smeri $-x$.

In [4]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.patches import Arc

# Podatki
L = 34.75   # mm
r = 17.5    # mm
Fa = -35.5  # N
Fr = 37.25  # N
Fc = 113.0  # N

print("Podatki naloge:")
print(f"L={L} mm, r={r} mm, Fa={Fa} N, Fr={Fr} N, Fc={Fc} N")

Podatki naloge:
L=34.75 mm, r=17.5 mm, Fa=-35.5 N, Fr=37.25 N, Fc=113.0 N


## 2) Redukcija sil v os stebla: zakaj to naredimo

Ker sili $F_r$ in $F_c$ delujeta na polmeru $r$, nista prijeti v osi stebla. Zato ob prenosu v os dodata momente.

Zakaj je to pomembno:
- brez tega bi podcenili torzijo in upogib,
- posledično bi bili NTM diagrami napačni.

Dodatna momenta zaradi ekscentričnosti:
- torzijski moment: $M_t^{(0)} = F_c r$
- dodatni upogibni člen: $M_{add} = F_r r$

In [5]:
Mt0 = Fc * r
Madd = Fr * r

print("Redukcija obremenitve v os stebla:")
print(f"Mt^(0) = Fc*r = {Fc:.2f}*{r:.2f} = {Mt0:.3f} N·mm")
print(f"Madd   = Fr*r = {Fr:.2f}*{r:.2f} = {Madd:.3f} N·mm")

Redukcija obremenitve v os stebla:
Mt^(0) = Fc*r = 113.00*17.50 = 1977.500 N·mm
Madd   = Fr*r = 37.25*17.50 = 651.875 N·mm


## 3) Metoda preseka: enačbe in funkcije notranjih količin

Izberemo presek na koordinati $s$, kjer je $0 \le s \le L$.
Ohranimo desni del konstrukcije in zapišemo ravnovesje.

Sile:
- $N(s) + F_a = 0 \Rightarrow N(s) = -F_a$
- $T_n(s) + F_r = 0 \Rightarrow T_n(s) = -F_r$
- $T_b(s) + F_c = 0 \Rightarrow T_b(s) = -F_c$

Momenti:
- $M_t(s) + F_c r = 0 \Rightarrow M_t(s) = -F_c r$
- $M_b(s) + F_r(L-s) = 0 \Rightarrow M_b(s) = -F_r(L-s)$
- $M_n(s) + F_c(L-s) + F_r r = 0 \Rightarrow M_n(s) = -F_c(L-s)-F_r r$

In [12]:
s = np.linspace(0, L, 500)

# Notranje sile in navori
N  = -Fa * np.ones_like(s)
Tn = -Fr * np.ones_like(s)
Tb = -Fc * np.ones_like(s)
Mt = -Fc * r * np.ones_like(s)
Mb = -Fr * (L - s)
Mn = -Fc * (L - s) - Fr * r

print("Mejne vrednosti:")
print(f"N   = {N[0]:+.2f} N")
print(f"Tn  = {Tn[0]:+.2f} N")
print(f"Tb  = {Tb[0]:+.2f} N")
print(f"Mt  = {Mt[0]:+.2f} N·mm")
print(f"Mb(0)={Mb[0]:+.2f} N·mm, Mb(L)={Mb[-1]:+.2f} N·mm")
print(f"Mn(0)={Mn[0]:+.2f} N·mm, Mn(L)={Mn[-1]:+.2f} N·mm")

Mejne vrednosti:
N   = +35.50 N
Tn  = -37.25 N
Tb  = -113.00 N
Mt  = -1977.50 N·mm
Mb(0)=-1294.44 N·mm, Mb(L)=-0.00 N·mm
Mn(0)=-4578.62 N·mm, Mn(L)=-651.88 N·mm


## 4) Skice, ki jih običajno zahteva naloga

V tem delu narišemo:
- geometrijsko skico konzole z vsemi zunanjimi silami,
- prosti diagram desnega dela pri preseku $s$,
- NTM diagrame ($N$, $T_n$, $T_b$, $M_t$, $M_b$, $M_n$),
- dodatno še skupni upogibni moment $M_{up}=\sqrt{M_b^2+M_n^2}$.

In [14]:
# Skica A (referencni slog): pokoncna in polezana oblika konstrukcije
figA, (axL, axR) = plt.subplots(1, 2, figsize=(14, 6))
for a in (axL, axR):
    a.set_aspect("equal")
    a.axis("off")

# -------------------------
# LEVA: "pokoncna" skica
# -------------------------
axL.set_xlim(-3, 6)
axL.set_ylim(-1, 11)

# navpicni element
axL.plot([0, 0], [0, 10], color="black", lw=4)
# vpetje zgoraj
axL.plot([-0.8, 0.8], [10, 10], color="black", lw=3)
for i in range(5):
    axL.plot([-0.8 + 0.35*i, -0.2 + 0.35*i], [10.6, 11.0], color="black", lw=1.4)

# spodnji lom in krajse vodoravno rame
elbow = (-1.1, 3.2)
axL.plot([0, -0.9], [2.4, 3.2], color="black", lw=4)
axL.plot([-0.9, 0], [3.2, 3.2], color="black", lw=4)
axL.plot(elbow[0], elbow[1], "ko", ms=4)

# kotna oznaka pri lomu
axL.plot([-0.55, -0.55], [2.55, 3.1], color="black", lw=1)
axL.plot([-0.55, -0.15], [2.55, 2.2], color="black", lw=1)

# dimenzija L
axL.plot([0.0, 4.5], [10, 10], color="black", lw=1.1)
axL.plot([0.0, 4.5], [2.4, 2.4], color="black", lw=1.1)
axL.annotate("", xy=(4.25, 9.9), xytext=(4.25, 2.5),
             arrowprops=dict(arrowstyle="-|>", lw=1.2, color="black"))
axL.annotate("", xy=(4.25, 2.5), xytext=(4.25, 9.9),
             arrowprops=dict(arrowstyle="-|>", lw=1.2, color="black"))
axL.text(3.6, 5.9, "$L$", fontsize=22)

# polmer r
axL.plot([0, 0], [0, 3.2], color="black", lw=1)
axL.annotate("", xy=(-0.95, 0.85), xytext=(-0.05, 0.15),
             arrowprops=dict(arrowstyle="-|>", lw=1.2, color="black"))
axL.annotate("", xy=(-0.05, 0.15), xytext=(-0.95, 0.85),
             arrowprops=dict(arrowstyle="-|>", lw=1.2, color="black"))
axL.text(-0.55, 1.0, "$r$", fontsize=16)

# sile v referencni orientaciji
axL.annotate("", xy=(-1.15, 3.2), xytext=(-2.1, 4.2),
             arrowprops=dict(arrowstyle="->", color="#1273c2", lw=2.8))
axL.text(-2.6, 3.35, "$F_r$", fontsize=19)

axL.annotate("", xy=(-0.95, 1.5), xytext=(-1.0, 3.2),
             arrowprops=dict(arrowstyle="->", color="#1273c2", lw=2.8))
axL.text(-1.65, 1.55, "$F_a$", fontsize=19)

axL.annotate("", xy=(2.0, 3.2), xytext=(-1.0, 3.2),
             arrowprops=dict(arrowstyle="->", color="#1273c2", lw=2.8))
axL.text(0.75, 3.45, "$F_c$", fontsize=22)

# -------------------------
# DESNA: "polezana" skica
# -------------------------
axR.set_xlim(-1, 12)
axR.set_ylim(-3, 6)

# nosilec
axR.plot([0, 9.2], [0, 0], color="black", lw=4)
# vpetje levo
axR.plot([0, 0], [-0.9, 0.9], color="black", lw=4)
for i in range(4):
    axR.plot([-0.75, -0.1], [0.4 - 0.35*i, -0.2 - 0.35*i], color="black", lw=1.3)

# lom na desnem koncu
axR.plot([9.2, 8.5], [0, -1.0], color="black", lw=4)
axR.plot([8.5, 10.5], [-1.0, -1.0], color="black", lw=1.8)
axR.plot([9.2, 11.2], [0.0, 0.0], color="black", lw=1.8)
axR.plot([9.25, 9.65], [-0.95, -0.45], color="black", lw=1)

# dimenzijska crta zgoraj
axR.plot([0, 9.2], [3.7, 3.7], color="black", lw=1)
axR.plot([0, 0], [0.0, 4.1], color="black", lw=1)
axR.plot([9.2, 9.2], [0.0, 4.1], color="black", lw=1)
axR.annotate("", xy=(0.2, 3.7), xytext=(9.0, 3.7),
             arrowprops=dict(arrowstyle="-|>", lw=1.2, color="black"))
axR.annotate("", xy=(9.0, 3.7), xytext=(0.2, 3.7),
             arrowprops=dict(arrowstyle="-|>", lw=1.2, color="black"))

# sile na lomu
joint = (8.5, -1.0)
axR.annotate("", xy=(joint[0]-0.2, joint[1]+0.1), xytext=(joint[0]-1.5, joint[1]-1.0),
             arrowprops=dict(arrowstyle="->", color="#1273c2", lw=2.8))
axR.annotate("", xy=(joint[0], 1.4), xytext=(joint[0], -1.0),
             arrowprops=dict(arrowstyle="->", color="#1273c2", lw=2.8))
axR.annotate("", xy=(joint[0]+2.3, joint[1]+0.05), xytext=(joint[0], joint[1]),
             arrowprops=dict(arrowstyle="->", color="#1273c2", lw=2.8))

axR.set_title("Skica A: kot v navodilih (pokoncno + polezano)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("DN1_skica_A_obremenitev.png", dpi=160, bbox_inches="tight")
plt.show()
print("Shranjeno: DN1_skica_A_obremenitev.png")

Shranjeno: DN1_skica_A_obremenitev.png


/tmp/ipykernel_10777/3169932519.py:100: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [15]:
# Skica B: prosti diagram v podobnem slogu (polezana verzija)
figB, ax = plt.subplots(figsize=(12, 5))
ax.set_xlim(-1.5, 12)
ax.set_ylim(-4, 5)
ax.set_aspect("equal")
ax.axis("off")

# nosilec
ax.plot([0, 9.2], [0, 0], color="black", lw=4)
ax.plot([0, 0], [-0.9, 0.9], color="black", lw=4)
for i in range(4):
    ax.plot([-0.75, -0.1], [0.4 - 0.35*i, -0.2 - 0.35*i], color="black", lw=1.3)

# presek pri s
xs = 4.0
ax.plot([xs, xs], [-1.2, 1.2], color="black", lw=2.2)
ax.text(xs-0.45, 1.45, "presek s", fontsize=10)

# dimenzije s in L-s
ax.annotate("", xy=(0, -3.0), xytext=(xs, -3.0), arrowprops=dict(arrowstyle="<->", color="gray", lw=1.2))
ax.annotate("", xy=(xs, -3.0), xytext=(9.2, -3.0), arrowprops=dict(arrowstyle="<->", color="gray", lw=1.2))
ax.text(xs/2, -3.55, "$s$", ha="center", fontsize=12, color="gray")
ax.text((xs+9.2)/2, -3.55, "$L-s$", ha="center", fontsize=12, color="gray")

# notranje velicine na preseku
ax.annotate("", xy=(xs-2.4, 0), xytext=(xs, 0), arrowprops=dict(arrowstyle="->", lw=2.8, color="blue"))
ax.text(xs-2.55, 0.35, "$N$", color="blue", fontsize=18)

ax.annotate("", xy=(xs, 2.1), xytext=(xs, 0), arrowprops=dict(arrowstyle="->", lw=2.8, color="red"))
ax.text(xs+0.2, 2.15, "$T_n$", color="red", fontsize=16)

ax.add_patch(plt.Circle((xs, 0), 0.22, color="green"))
ax.plot(xs, 0, "w.", ms=5)
ax.text(xs-0.55, -0.7, "$T_b$", color="green", fontsize=16)

arc = Arc((xs, 0), 1.8, 1.8, theta1=20, theta2=340, color="darkorange", lw=2.8)
ax.add_patch(arc)
ax.annotate("", xy=(xs+0.8*np.cos(np.radians(20)), 0.8*np.sin(np.radians(20))),
            xytext=(xs+0.8*np.cos(np.radians(40)), 0.8*np.sin(np.radians(40))),
            arrowprops=dict(arrowstyle="->", lw=1.8, color="darkorange"))
ax.text(xs-1.15, 1.7, "$M_t$", color="darkorange", fontsize=18)

ax.annotate("", xy=(xs, 1.4), xytext=(xs, 0.4), arrowprops=dict(arrowstyle="->", lw=2.2, color="purple"))
ax.text(xs+0.25, 1.45, "$M_b$", color="purple", fontsize=18)

ax.add_patch(plt.Circle((xs, 0), 0.38, edgecolor="brown", facecolor="none", lw=2.2))
ax.plot([xs-0.25, xs+0.25], [0, 0], color="brown", lw=2)
ax.plot([xs, xs], [-0.25, 0.25], color="brown", lw=2)
ax.text(xs-1.7, -0.1, "$M_n$", color="brown", fontsize=18)

# zunanji sili na desnem koncu
joint = (9.2, 0)
ax.plot([9.2, 9.2], [0, 2.7], color="#1273c2", lw=1.7, ls="--")
ax.annotate("", xy=(8.0, 0), xytext=(9.2, 0), arrowprops=dict(arrowstyle="->", lw=2.8, color="#1273c2"))
ax.annotate("", xy=(9.2, 2.9), xytext=(9.2, 0.7), arrowprops=dict(arrowstyle="->", lw=2.8, color="#1273c2"))
ax.add_patch(plt.Circle((9.2, 2.7), 0.22, color="#1273c2"))
ax.plot(9.2, 2.7, "w.", ms=5)
ax.text(8.15, 0.35, "$F_a$", color="#1273c2", fontsize=15)
ax.text(9.35, 2.85, "$F_r$", color="#1273c2", fontsize=15)
ax.text(9.45, 2.45, "$F_c$", color="#1273c2", fontsize=15)

ax.set_title("Skica B: prosti diagram (slog kot v navodilih)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("DN1_skica_B_prosti_diagram.png", dpi=160, bbox_inches="tight")
plt.show()
print("Shranjeno: DN1_skica_B_prosti_diagram.png")

Shranjeno: DN1_skica_B_prosti_diagram.png


/tmp/ipykernel_10777/2995760213.py:65: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5) NTM diagrami in grafični zapis računov

Spodnji diagrami so neposreden grafični zapis funkcij:
- $N(s)$, $T_n(s)$, $T_b(s)$ so konstantni,
- $M_t(s)$ je konstanten,
- $M_b(s)$ in $M_n(s)$ sta linearna po $s$.

In [9]:
fig = plt.figure(figsize=(14, 16))
fig.suptitle(
    "NTM diagrami - celotna naloga\n"
    f"L={L} mm, r={r} mm, Fa={Fa} N, Fr={Fr} N, Fc={Fc} N",
    fontsize=13, fontweight="bold"
)

gs = gridspec.GridSpec(3, 2, figure=fig, hspace=0.5, wspace=0.32)
plots = [
    (N,  "N(s) [N]",            "Aksialna sila N",         "tab:blue"),
    (Tn, "Tn(s) [N]",           "Strižna sila Tn",         "tab:orange"),
    (Tb, "Tb(s) [N]",           "Strižna sila Tb",         "tab:green"),
    (Mt, "Mt(s) [N·mm]",        "Torzija Mt",              "tab:red"),
    (Mb, "Mb(s) [N·mm]",        "Upogibni moment Mb",      "tab:purple"),
    (Mn, "Mn(s) [N·mm]",        "Upogibni moment Mn",      "tab:brown"),
]

for i, (y, yl, ttl, c) in enumerate(plots):
    ax = fig.add_subplot(gs[i // 2, i % 2])
    ax.plot(s, y, color=c, lw=2)
    ax.fill_between(s, y, 0, color=c, alpha=0.15)
    ax.axhline(0, color="black", lw=0.8)
    ax.set_xlim(0, L)
    ax.set_xlabel("s [mm]")
    ax.set_ylabel(yl)
    ax.set_title(ttl, fontsize=10, fontweight="bold")
    ax.grid(True, alpha=0.3, linestyle="--")

    y0, yL = float(y[0]), float(y[-1])
    d = max(abs(y0), abs(yL), 1.0) * 0.08
    ax.text(0.4, y0 + (d if y0 >= 0 else -d), f"{y0:.1f}", color=c, fontsize=8)
    ax.text(L - 0.4, yL + (d if yL >= 0 else -d), f"{yL:.1f}", color=c, fontsize=8, ha="right")

plt.savefig("DN1_NTM_diagrami_celotna.png", dpi=150, bbox_inches="tight")
plt.show()
print("Shranjeno: DN1_NTM_diagrami_celotna.png")

Shranjeno: DN1_NTM_diagrami_celotna.png


/tmp/ipykernel_10777/3898480653.py:35: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6) Verifikacija in končni povzetek

Za preverbo izračunamo skupni upogibni moment:
$$M_{up}(s)=\sqrt{M_b(s)^2 + M_n(s)^2}$$

Najbolj obremenjeno mesto pri konzoli je praviloma vpetje ($s=0$), zato tam preverimo največje vrednosti.

In [10]:
Mup = np.sqrt(Mb**2 + Mn**2)

fig2, ax = plt.subplots(figsize=(8, 4))
ax.plot(s, Mup, color="navy", lw=2)
ax.fill_between(s, Mup, 0, color="navy", alpha=0.15)
ax.axhline(0, color="black", lw=0.8)
ax.set_xlim(0, L)
ax.set_xlabel("s [mm]")
ax.set_ylabel("Mup [N·mm]")
ax.set_title("Skupni upogibni moment Mup(s)", fontweight="bold")
ax.grid(True, alpha=0.3, linestyle="--")
plt.tight_layout()
plt.savefig("DN1_Mup_celotna.png", dpi=150, bbox_inches="tight")
plt.show()

print("Končni povzetek:")
print(f"N = {N[0]:.2f} N")
print(f"Tn = {Tn[0]:.2f} N")
print(f"Tb = {Tb[0]:.2f} N")
print(f"Mt = {Mt[0]:.2f} N·mm")
print(f"Mb(0) = {Mb[0]:.2f} N·mm, Mb(L) = {Mb[-1]:.2f} N·mm")
print(f"Mn(0) = {Mn[0]:.2f} N·mm, Mn(L) = {Mn[-1]:.2f} N·mm")
print(f"Mup(0) = {Mup[0]:.2f} N·mm")
print(f"Mt/Mup(0) = {abs(Mt[0]) / Mup[0]:.3f}")

Končni povzetek:
N = 35.50 N
Tn = -37.25 N
Tb = -113.00 N
Mt = -1977.50 N·mm
Mb(0) = -1294.44 N·mm, Mb(L) = -0.00 N·mm
Mn(0) = -4578.62 N·mm, Mn(L) = -651.88 N·mm
Mup(0) = 4758.09 N·mm
Mt/Mup(0) = 0.416


/tmp/ipykernel_10777/2773305385.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7) Kaj oddati

Ta dokument že vsebuje vse, kar se pri taki nalogi običajno zahteva:
- skico geometrije in zunanjih sil,
- prosti diagram telesa pri preseku,
- analitične enačbe z razlago,
- numerične rezultate,
- NTM diagrame,
- dodatni diagram skupnega upogibnega momenta.

Datoteke slik, ki se ustvarijo ob zagonu:
- DN1_skica_A_obremenitev.png
- DN1_skica_B_prosti_diagram.png
- DN1_NTM_diagrami_celotna.png
- DN1_Mup_celotna.png